# Step 21.2 — Greedy vs sampling：选择 token，也传递随机 key

在仓库根目录先同步母版：

```bash
git pull
```

再用时间戳建立独立工作副本，不覆盖笔记（仓库已有 notebooks/work 目录）：

```bash
cp -n notebooks/93_step21_2_greedy_vs_sampling.ipynb notebooks/work/93_step21_2_greedy_vs_sampling_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

命令仅作未来参考，本次未执行；同名副本存在时 cp -n 不覆盖。环境沿用项目 Python 3.12 / .venv，MLX API 固定 **v0.32.2**。

2026-11-17 教学修订：**21.2 累计 1/2＝历史修正 0/2＋本次教学修订 1 次**。两段原代码逐字保留，只补讲解与状态跟踪，不修改 21.1/21.3。所有代码单元 execution_count=null、outputs=[]；没有运行、采样、训练或读取语料，纸算不是程序输出。


## 1. 只改变 21.1 的“选择”环节

[21.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/4d2f65011a683a9bde88c75e2b90ee6267a59c0b/notebooks/92_step21_1_generation_loop.ipynb)已经讲过 context → 最后位置 logits → 选一个 ID → append。本节保留这个顺序，只比较两种选择规则：

| 规则 | 从同一行 logits 得到什么？ | 是否需要随机性？ |
|---|---|---|
| greedy（贪心选择） | 最大分数所在的 token ID | 不需要 |
| categorical sampling（类别分布抽样） | 按各候选概率抽一个 token ID | 需要显式 random key |

logit 是未归一化分数，不是 token ID，也不直接是概率。换选择规则不更新模型权重；sampling 允许非最大候选出现，也不自动保证更高质量或事实性。逐步 greedy 不保证整串序列的联合概率最大。

为了只观察选择器，本节固定同一行人工 logits。真实生成每次 append 后前缀改变，下一轮 logits 也可能改变，不能把固定分布的重复抽样直接等同于完整语言模型生成。


## 2. 从分数到概率：最少数学前置

categorical distribution 给离散候选 i 分配概率 pᵢ≥0，且总和为 1。softmax 用指数把实数分数变成正权重，再除以总和：

$$
m=\max_j z_j,\qquad
p_i=\frac{\exp(z_i-m)}{\sum_j\exp(z_j-m)}.
$$

exp 是自然指数函数，ln 是它的反函数，例如 exp(ln 2)=2。减去同一个 m，相当于所有指数权重同乘 exp(−m)，分子分母抵消；在精确算术下概率不变，又能避免直接计算过大的正指数。有限精度下仍可能舍入或下溢，不承诺无限精度。

greedy 只需比较分数；softmax 保持它们的大小顺序，无需先算概率再 argmax。MLX v0.32.2 的 `mx.random.categorical` 也直接接受 **logits**，不能把概率数组再传进去当 logits，否则会对概率再次做指数权重，得到不同分布。后面的 probabilities 只用于未来显示，不是抽样函数的输入。


## 3. 手算同一行分数，不预测某个 seed 的随机结果

令 z=[ln 2, ln 3, ln 5]，三个位置就是 ID 0、1、2。最大值为 ln 5，减去它之后的指数权重是：

$$
[2/5,\ 3/5,\ 1],\qquad
\text{总和}=2,\qquad
p=[0.2,\ 0.3,\ 0.5].
$$

因此 greedy 选 **ID 2**，不是返回分数 ln 5。sampling 有 0.2 的概率选 ID 0、0.3 的概率选 ID 1、0.5 的概率选 ID 2。

可以把 [0,1) 想象成长度分别为 0.2、0.3、0.5 的三个区间：一个假定落在 0.35 的点位于第二段，对应 ID 1。这里 0.35 是为解释分布而指定的纸面值，不是 seed=7 产生的数，也不是在说明 MLX 的内部抽样算法。

若在同一理论分布下独立抽两次，两次都选 ID 2 的概率是 0.5×0.5=0.25。重复 token 并不说明 sampling 退化成 greedy；本节也不声称实际 key 会产生哪几个 token。源代码中的 float32 logits 是这些数学数值的有限精度近似。


## 4. 先核对 shape 和两个返回值

`axis=-1` 表示最后一轴是候选类别。未指定 shape 或 num_samples 时，categorical 的输出删除这一轴：

| 输入 logits | 选择器输出 token | 意义 |
|---|---|---|
| (B,V) | (B,) | 每行选一个原词表 ID |
| 本节 (1,3) | (1,) | 一个 batch 行的一个 ID |
| 单独一行 (V,) | () | 标量数组；不能直接冒充带 batch 轴的接口 |

categorical 原生输出 uint32；这里两个 helper 都显式转为 int32，以接上 21.1 的 IDs。一个 key 可以驱动整次 (B,V) 抽样调用，输出 B 个 ID；它不是“每个 batch 行必须额外手动 split 一次”的接口。本节观察代码仍专用 B=1，后面的 token.item() 也要求只有一个元素。

`mx.random.key(seed)` 创建 shape (2,) 的 uint32 key。`mx.random.split(key)` 默认 num=2，返回 shape (2,2)；Python 解包后 next_key 和 draw_key 各为 (2,)。这不是 token 的 shape，两个子 key 也不表示抽两个 token。

下面两个 helper 已足够短，保留同一代码块。先看函数签名与最后一行：choose_greedy 返回 token；choose_sample 返回 **(token, next_key)**，不是 (next_key, token)。


In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

def choose_greedy(logits):
    return mx.argmax(logits, axis=-1).astype(mx.int32)

def choose_sample(logits, key):
    next_key, draw_key = mx.random.split(key)
    token = mx.random.categorical(logits, axis=-1, key=draw_key)
    return token.astype(mx.int32), next_key

logits = mx.array([[math.log(2), math.log(3), math.log(5)]], dtype=mx.float32)

## 5. 一次抽样：拆分一次，把“剩余状态”交回调用者

random key 是显式的伪随机输入。这里说“消费 key”，指调用者约定每次抽样使用新的分支，并将保留分支交给下一轮；函数不会原地修改传入的 key 数组。

设调用前 key 为 k₀，逐行对照 choose_sample：

| 源码动作 | 纸面状态变化 | 为什么需要？ |
|---|---|---|
| next_key, draw_key = split(k₀) | 得到 k₁ 与 d₀ | k₁ 留给下一次，d₀ 仅用于此次抽样 |
| categorical(logits, key=d₀) | 得到 token t₀，shape (B,) | 用当前分布与本次随机分支选择 |
| return token.astype(int32), k₁ | 返回 (t₀,k₁) | 同时交付结果与后续所需状态 |
| 调用方 token, key = choose_sample(logits, key) | token 绑定 t₀；key 改绑 k₁ | 下一次不能仍使用旧变量值 k₀ |

Python 先算右侧调用，再按顺序把返回值赋给左侧名字。helper 内 next_key / draw_key 的解包顺序，和它最后 return 的顺序并不相同；必须以 return 为准。

本函数一次调用只有一次 split 和一次 categorical。split 不会替你抽样；categorical 也不会替你返回新的 key，所以 helper 明确返回 k₁。


## 6. 两次调用：第二次拆 k₁，不是再拆 k₀

下面是符号跟踪，不填写任何随机 token 值，也不计算实际 key：

| 调用 | 传入 key | split 的两个分支 | 此次抽样使用 | 返回给下一次 |
|---:|---|---|---|---|
| 第一次 | k₀ | (k₁,d₀) | d₀ | (t₀,k₁) |
| 第二次 | k₁ | (k₂,d₁) | d₁ | (t₁,k₂) |

两次调用合计两次 split；不是每一个 token 都先后 split 两次。t₀ 与 t₁ 可以相同，新的随机分支不保证 token 不重复。

相反，若丢弃返回的 next_key，第二次仍传 k₀，则会重新得到相同的两个分支。对相同 logits、shape 和同一实现中的相同执行条件，这在重复同一个伪随机选择，不是获得一次新的抽样。若 logits 已变，即使 key 重用也不保证 token 相同，但随机性仍被复用，不能当作独立新抽样。

固定初始 seed 与相同的 key 传递顺序有助于复现；不保证跨版本、硬件或数值路径逐位一致。不要在循环每一轮都重新 key(7)，那会反复回到同一个起点。


## 7. 哪些操作完全不需要随机 key？

- **choose_greedy**没有 key 参数，只做 argmax 与 dtype 转换；无需创建、split 或更新 key。
- 为讲解计算 **softmax probabilities**、取最后位置、增轴、append，都不引入随机抽样。
- 没有调用选择器的路径，例如未来生成 0 个新 token，不发生抽样或 split。外围若无条件先 split，那是外围额外推进了 key，并非 greedy 的要求。

传显式 draw_key 的 categorical 使用这一个 key，不会推进默认全局随机序列。key(seed) 与 split(显式 key) 也不向默认序列索取新 key。省略 categorical 的 key 参数才会使用默认 key；本节 helper 没有省略。

因此应用层要保持的状态是自己返回并重绑定的 key。`mx.eval(token,key)` 是求值边界，不是“再抽一次”或自动把 key 换成另一个分支。

下一节 [21.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/4d2f65011a683a9bde88c75e2b90ee6267a59c0b/notebooks/94_step21_3_temperature.ipynb)的 temperature_logits(logits, temperature) 只返回缩放后的 logits，没有随机 key；它和本节“选择并传递 key”的职责仍分开。这里不展开温度实现。


## 8. 接回 21.1：选择器输出 (B,)，append 需要 (B,1)

[21.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/4d2f65011a683a9bde88c75e2b90ee6267a59c0b/notebooks/92_step21_1_generation_loop.ipynb)的 last_logits 返回 (B,V)，实际单条 prompt 接口中 B=1。本节换的是下表的中间一行；选择器本身不调用模型、不保存 history，也不会 append：

| 阶段 | greedy 路径 | sampling 路径 |
|---|---|---|
| 从当前前缀取得分数 | last，shape (B,V) | 同一接口取得 last |
| 选择 | token = choose_greedy(last) | token, key = choose_sample(last, key) |
| 准备拼接 | token[:,None]，shape (B,1) | 相同增轴；只用 token，不能把整个返回 tuple 拼接 |
| 追加后保留状态 | 新 ids，shape (B,T+1) | 新 ids **和返回的 key** |

None 插入的是长度为 1 的时间轴，不增加一个 batch。沿 axis=1 拼接保留已有前缀，然后下一轮再从新的 ids 求 last。

对 sampling 而言，循环状态可以纸面记作 (ids_r,k_r) → (ids_(r+1),k_(r+1))；greedy 的选择环节只需传 ids。已有 generate_greedy 的签名没有 key，也不会接受 choose_sample 返回的 tuple；上表说明接入关系，**本轮不改造 21.1，也不增加一个新生成器**。

完整循环仍需原来的 prompt、shape、容量与输入检查。helper 对 (B,V) 的选择不意味着原 B=1 生成函数突然成了完整 batch/EOS/cache 生成器。


## 9. 八次采样：仅供未来观察，不是概率验证

下面保留原观察单元，当前没有运行记录。它先算 probabilities 供显示，greedy_id 用原 logits；然后用 key(7) 初始化一次，在八次循环中反复接收返回的 key。每次传入的是同一行 logits，**没有把 token 追加成新前缀**。

mx.eval(token,key) 后，token.item() 把本例唯一的 ID 取成 Python 数加入 draws；这里不能把批量多个 ID 的结果直接 item()。最后打印的八个数只能是一次未来观察，不是概率恰好等于 [0.2,0.3,0.5] 的验证，也不是训练质量测试。

运行前可以预测 shape、key 传递与理论分布，不能把 seed=7 的具体八个 ID 当作本页已知结果。本轮不补写任何输出。


In [ ]:
probabilities = mx.softmax(logits, axis=-1)
greedy_id = choose_greedy(logits)
key = mx.random.key(7)
draws = []
for _ in range(8):
    token, key = choose_sample(logits, key)
    mx.eval(token, key)
    draws.append(token.item())
mx.eval(probabilities, greedy_id)
print("p:", probabilities.tolist(), "greedy:", greedy_id.tolist())
print("eight draws, not a frequency estimate:", draws)

## 10. 输入与确定性边界

本节人工 logits 全部有限。两个 helper 很短，**没有自行实现完整输入校验**；使用者需保证最后一轴是候选、每行至少有一个有限分数且没有 NaN/+Inf。后续 mask 可把部分候选设为 −Inf，但全行 −Inf 没有合法归一化分布，须在上游拒绝。

21.1 的 last_logits 做 shape 与有限值检查；不能绕过它后仍声称新输入已被检查。本节不实现 mask、top-k 或 context cropping。

greedy 在固定输入和数值路径下不需要随机性，但并列最大值仍需约定 tie policy；这里不把并列时的具体索引选择作为跨实现契约，也不把它解释成模型的“偏好”。


## 11. 练习（不附答案）

1. 对 z=[0,0,ln2] 计算概率并比较 greedy 与 sampling 的可能输出。
2. 解释为何重复八次结果不能验证概率恰好等于理论值。
3. 如果希望生成两条独立序列，应该如何管理各自 key？画出 key 流程，不给训练代码。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。

## 固定版本依据与材料状态

- [MLX v0.32.2 Python random bindings](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/random.cpp)：split 默认 num=2，categorical 的 logits、axis、shape、dtype 与显式／默认 key 分支。
- [MLX v0.32.2 random 实现](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/random.cpp)：key 的 uint32 (2,) 表示、split 的 (num,2) 输出及 categorical 的轴处理。

本次只补足概率语义、一次／两次 key 传递和 append 接口；两段原代码的内容与行为保持，练习不代答。所有代码单元未执行、无输出，仅做 JSON/schema、合法唯一 ID、语法与源码静态核验，没有运行课程代码、采样、测试、训练、实验、读取语料或 checkpoint。

21.2 累计 **1/2**（历史 0/2＋本次教学修订 1 次）；其他历史计数保留。材料通过静态核验不等于学员掌握或运行通过。本轮停在 21.2，等待独立验收。
